#### Imports

In [141]:
import pandas as pd
import matplotlib.pyplot as plt

#### Import data

In [142]:
train = pd.read_csv('traindata_total.csv')
train = train.drop(columns = ['index', 'Service:RDT-ID'])
train.head()

,Service:Date,Service:Type,Service:Company,Service:Train number,Service:Completely cancelled,Service:Partly cancelled,Service:Maximum delay,Stop:RDT-ID,Stop:Station code,Stop:Station name,Stop:Arrival time,Stop:Arrival delay,Stop:Arrival cancelled,Stop:Departure time,Stop:Departure delay,Stop:Departure cancelled,Stop:Platform change,Stop:Planned platform,Stop:Actual platform
0,2025-12-01,Intercity,NS,1410,False,False,0,156552404,UT,Utrecht Centraal,2025-12-01T03:48:00+01:00,7.0,False,NaN,NaN,NaN,False,18.0,18.0
1,2025-12-01,Intercity,NS,1409,False,True,0,156552405,UT,Utrecht Centraal,NaN,NaN,NaN,2025-12-01T02:16:00+01:00,0.0,True,False,NaN,NaN
2,2025-12-01,Intercity,NS,1414,False,False,0,156552418,UT,Utrecht Centraal,2025-12-01T04:48:00+01:00,0.0,False,NaN,NaN,NaN,False,18.0,18.0
3,2025-12-01,Nightjet,NS Int,402,False,True,0,156552440,UT,Utrecht Centraal,2025-12-01T08:53:00+01:00,100.0,False,2025-12-01T08:55:00+01:00,100.0,False,False,14.0,14.0
4,2025-12-01,Intercity,NS,1413,False,False,26,156552444,UT,Utrecht Centraal,NaN,NaN,NaN,2025-12-01T03:16:00+01:00,7.0,False,False,18.0,18.0


#### Data cleaning

In [143]:
train.columns = train.columns.str.replace('Service:', '', regex=False)
train.columns = train.columns.str.replace('Stop:', '', regex=False)
train.columns = train.columns.str.replace('RDT-ID', 'Train ID', regex=False)
train.columns = train.columns.str.replace('Arrival time', 'Actual Arrival Time', regex=False)
train.columns = train.columns.str.replace('Departure time', 'Actual Departure Time', regex=False)

cols_drop = ['Completely cancelled', 'Partly cancelled','Maximum delay','Train number', 'Station name', 'Station code','Platform change', 'Planned platform', 'Actual platform', 'Company']
cols_dt = ['Actual Arrival Time', 'Actual Departure Time']

train = train.drop(columns = cols_drop)
train[cols_dt] = train[cols_dt].apply(
    lambda col: pd.to_datetime(col, utc = True).dt.strftime('%H:%M')
)

In [144]:
# Convert actual arrival/departure times to datetime
train['Actual Arrival Time'] = pd.to_datetime(
    train['Actual Arrival Time'], format='%H:%M'
)

train['Actual Departure Time'] = pd.to_datetime(
    train['Actual Departure Time'], format='%H:%M'
)

# Calculate scheduled times by subtracting the delay
train['Scheduled Arrival Time'] = (
    train['Actual Arrival Time']
    - pd.to_timedelta(train['Arrival delay'], unit='m')
)

train['Scheduled Departure Time'] = (
    train['Actual Departure Time']
    - pd.to_timedelta(train['Departure delay'], unit='m')
)

# Convert all time columns back to HH:MM
time_cols = [
    'Actual Arrival Time',
    'Actual Departure Time',
    'Scheduled Arrival Time',
    'Scheduled Departure Time'
]

train[time_cols] = train[time_cols].apply(
    lambda col: col.dt.strftime('%H:%M')
)

In [145]:
train.head()

,Date,Type,Train ID,Actual Arrival Time,Arrival delay,Arrival cancelled,Actual Departure Time,Departure delay,Departure cancelled,Scheduled Arrival Time,Scheduled Departure Time
0,2025-12-01,Intercity,156552404,02:48,7.0,False,NaN,NaN,NaN,02:41,NaN
1,2025-12-01,Intercity,156552405,NaN,NaN,NaN,01:16,0.0,True,NaN,01:16
2,2025-12-01,Intercity,156552418,03:48,0.0,False,NaN,NaN,NaN,03:48,NaN
3,2025-12-01,Nightjet,156552440,07:53,100.0,False,07:55,100.0,False,06:13,06:15
4,2025-12-01,Intercity,156552444,NaN,NaN,NaN,02:16,7.0,False,NaN,02:09


In [ ]:
delayed_trains = (len(train[(train['Arrival delay'] == True) | (train['Departure delay'] == True)])) / len(train) * 100
total_cancellations = len(train[(train['Arrival cancelled'] == True) & (train['Departure cancelled'] == True)])
total_cancellations_perct = total_cancellations / len(train) * 100


arrival_ids = set(train.loc[train['Arrival cancelled'] == True, 'Train ID'])
departure_ids = set(train.loc[train['Departure cancelled'] == True, 'Train ID'])

arrival_only = arrival_ids - departure_ids
departure_only = departure_ids - arrival_ids

print(f"Number of partial (arrival) cancellations: {len(arrival_only)}")
print(f"Number of partial (departure) cancellations: {len(departure_only)}")
print(f"Number of complete cancellations: {total_cancellations}")
print("")
print(f"{delayed_trains:.1f}% of trains were delayed.")
print(f"{total_cancellations_perct:.1f}% of trains were completely cancelled.")

Number of partial (arrival) cancellations: 4230
Number of partial (departure) cancellations: 4007
Number of complete cancellations: 10082

10.6% of trains were delayed.
5.2% of trains were completely cancelled.


- Analyse train delays (amount, spread) over time
- Analyse train cancellations over time
- Classify primary and secondary delays (or a variant on it on incoming vs outgoing delays)


- Current max delays are max delay over all stops for the trajectory

#### Data visualization

#### Data manipulation & Analysis